# SepsisGuard — separate Qwen-only LoRA prediction experiment

Choose **A100 80GB / high RAM** and **Run all**. This notebook fine-tunes Qwen itself on the original A/train split, selects a completed adapter using A/validation, calibrates using A/calibration, then runs an exploratory matched B pilot. Original notebooks, models and results remain unchanged on `main`; this notebook lives on `experiment/qwen-prediction-lora-v1`.

The bounded first experiment uses 512 training endpoints for two epochs. LoRA adapts the final 16 language-model layers; it does not update all 27B base parameters. Numerical predictions come from Qwen's own next-token scores for `0` and `1`. No LSTM/logistic predictions enter Qwen's inputs.

Validation, calibration and B pilot counts are 256, 512 and 1,024 sampled hours. Uniform class-stratified sampling with inverse-inclusion weights permits an exploratory estimate at the original hourly prevalence. Both pretrained Qwen and fine-tuned Qwen are scored, together with saved original-model predictions for exactly the same B hours. This is a pilot, not a full-cohort comparison. B has already been evaluated; confirmatory evaluation needs a new independent cohort.

Only past/current observations enter prompts. Labels already include the benchmark's six-hour shift and are preserved. Sparse sampling does not support alarm-episode, official timing utility or true early-warning claims.

Private inputs, adapters, optimizer checkpoints and predictions persist under `MyDrive/SepsisGuard-UAR/experiments/qwen_prediction/<run-id>`. A 90-minute compute budget after loading returns an explicit paused status with resumable checkpoints if reached. Read `QWEN_RESULT.json` and `QWEN_REPORT.md`; improvement is measured, never assumed.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess
import torch
from google.colab import drive
assert torch.cuda.is_available(), 'Choose A100 80GB / high RAM.'
print('GPU:',torch.cuda.get_device_name(0),'VRAM GiB:',round(torch.cuda.get_device_properties(0).total_memory/2**30,2),flush=True)
assert torch.cuda.get_device_properties(0).total_memory/2**30 >=70, 'This BF16 27B profile requires an 80GB GPU.'
drive.mount('/content/drive')
DRIVE_ROOT=Path('/content/drive/MyDrive/SepsisGuard-UAR')
assert DRIVE_ROOT.is_dir(), 'Mount the Drive containing the completed preparation checkpoint.'
print('New private output namespace:',DRIVE_ROOT/'experiments/qwen_prediction')


In [ ]:
BRANCH='experiment/qwen-prediction-lora-v1'
REPO_URL='https://github.com/shakiful/Sepsisguard-Uar.git'
PROJECT=Path('/content/sepsisguard-qwen-prediction')
if not PROJECT.exists():
    subprocess.run(['git','clone','--single-branch','--branch',BRANCH,REPO_URL,str(PROJECT)],check=True)
else:
    assert (PROJECT/'.git').is_dir(), 'Unexpected source directory; preserve it and choose a new project path.'
    assert subprocess.check_output(['git','-C',str(PROJECT),'remote','get-url','origin'],text=True).strip().lower()==REPO_URL.lower()
    assert not subprocess.check_output(['git','-C',str(PROJECT),'status','--porcelain','--untracked-files=no'],text=True).strip(), 'Modified source detected; preserve it before updating.'
    subprocess.run(['git','-C',str(PROJECT),'fetch','origin',BRANCH],check=True)
    subprocess.run(['git','-C',str(PROJECT),'merge','--ff-only','FETCH_HEAD'],check=True)
os.chdir(PROJECT)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[dev,download,agents]','-r','experiments/qwen_prediction/requirements.txt'],check=True)
subprocess.run([sys.executable,'-m','experiments.qwen_prediction.prepare_runtime'],check=True)
print('Separate experiment source:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
subprocess.run([sys.executable,'-m','pytest','-q','experiments/qwen_prediction/tests'],check=True)
MODEL_CACHE=Path('/content/sepsisguard-qwen-model-cache')
DATA_ROOT=PROJECT/'data/raw'
for old_data in (Path('/content/sepsisguard-uar-unified/data/raw'),Path('/content/sepsisguard-uar/data/raw')):
    if old_data.is_dir():
        DATA_ROOT=old_data
        break


In [ ]:
command=[sys.executable,'-u','-m','experiments.qwen_prediction.run',
         '--data-root',str(DATA_ROOT),'--storage-root',str(DRIVE_ROOT),
         '--model-cache',str(MODEL_CACHE),'--max-minutes','90']
process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in process.stdout:
    print(line,end='',flush=True)
return_code=process.wait()
if return_code==75:
    print('Compute budget reached. Verified adapter/optimizer/prediction checkpoints are saved. Run all again resumes this isolated experiment.')
elif return_code:
    raise RuntimeError('Qwen experiment stopped. Inspect the error; the original experiment and existing private checkpoints are preserved.')
else:
    print('Qwen fine-tuning and exploratory paired evaluation completed. Read the saved aggregate report.')


In [ ]:
from IPython.display import display, Markdown, SVG
results=list((DRIVE_ROOT/'experiments/qwen_prediction').glob('qwen-lora-*/QWEN_RESULT.json'))
if results:
    result_path=max(results,key=lambda p:p.stat().st_mtime)
    result=json.loads(result_path.read_text())
    print('Completed isolated run:',result['run_id'])
    display(Markdown((result_path.parent/'QWEN_REPORT.md').read_text()))
    display(SVG(filename=str(result_path.parent/'AGGREGATE_CURVES.svg')))
else:
    print('No completed real comparison yet. Review PROGRESS.json or the execution error; no improvement result is being claimed.')
